In [1]:
import os
import json
import time
from typing import List
import sys

sys.path.append('Extract_kg')

from config import ENTITY_FILE, OUTPUT_JSON, INPUT_FILES
from entity_processor import load_existing_entities
from text_processor import read_source_files
from kg_builder import KnowledgeGraphBuilder
from output_manager import analyze_knowledge_graph

In [2]:
def create_knowledge_graph(file_paths: List[str], output_json_path: str):
    """Main execution function."""
    print("KNOWLEDGE GRAPH EXTRACTION FROM EXISTING ENTITIES")
    print("Using multi-phase relationship extraction with entity coverage\n")
    
    entities = load_existing_entities(ENTITY_FILE)
    if not entities:
        print("No entities found. Exiting.")
        return None
    
    source_files = read_source_files(file_paths)
    print(f"Read {len([f for f in source_files.values() if f])} source files with content")
    
    if os.path.exists(output_json_path):
        with open(output_json_path, 'r', encoding='utf-8') as f:
            existing_kg = json.load(f)
        print(f"Loaded existing KG: {len(existing_kg.get('entities', []))} entities, {len(existing_kg.get('triplets', []))} triplets\n")
    else:
        existing_kg = {'entities': entities, 'triplets': []}
        print("Starting fresh KG\n")
    
    start_time = time.time()
    
    builder = KnowledgeGraphBuilder()
    kg = builder.build_from_existing(entities, source_files)
    
    end_time = time.time()
    
    print(f"\nCompleted knowledge graph construction in {end_time - start_time:.2f} seconds")
    
    try:
        with open(output_json_path, 'w', encoding='utf-8') as f:
            json.dump(kg.dict(), f, ensure_ascii=False, indent=2)
        print(f"Saved knowledge graph to: {output_json_path}")
    except Exception as e:
        print(f"Error saving file: {e}")
        try:
            simple_kg = {
                'entities': [entity.dict() for entity in kg.entities],
                'triplets': [triplet.dict() for triplet in kg.triplets]
            }
            with open(output_json_path, 'w', encoding='utf-8') as f:
                json.dump(simple_kg, f, ensure_ascii=False, indent=2)
            print(f"Saved using alternative method: {output_json_path}")
        except Exception as e2:
            print(f"Alternative save also failed: {e2}")
    
    analyze_knowledge_graph(kg)
    
    connected_entities = set()
    for triplet in kg.triplets:
        connected_entities.add(triplet.subject_id)
        connected_entities.add(triplet.object_id)
    
    unconnected_entities = [
        entity.dict() for entity in kg.entities 
        if entity.id not in connected_entities
    ]
    
    if unconnected_entities:
        with open("unconnected_entities.json", 'w', encoding='utf-8') as f:
            json.dump(unconnected_entities, f, ensure_ascii=False, indent=2)
        print(f"\nSaved {len(unconnected_entities)} unconnected entities to: unconnected_entities.json")
    
    return kg

if __name__ == "__main__":
    kg = create_knowledge_graph(INPUT_FILES, OUTPUT_JSON)

KNOWLEDGE GRAPH EXTRACTION FROM EXISTING ENTITIES
Using multi-phase relationship extraction with entity coverage

Đã tải 490 thực thể từ file
Read 17 source files with content
Starting fresh KG

Bắt đầu xây dựng knowledge graph với xử lý theo chủ đề...
Đã tạo lookup cho 607 thực thể

Phát hiện 6 chủ đề:
  Chủ đề 1: 3 file
  Chủ đề 2: 2 file
  Chủ đề 3: 4 file
  Chủ đề 4: 2 file
  Chủ đề 5: 3 file
  Chủ đề 6: 3 file

XỬ LÝ CHỦ ĐỀ: Chủ đề 1

XỬ LÝ CHỦ ĐỀ 1: THẾ GIỚI TRONG VÀ SAU CHIẾN TRANH LẠNH
Trọng tâm: Chủ đề tập trung vào cấu trúc quyền lực thế giới, các tổ chức quốc tế, quan hệ giữa các cường quốc, ...
Đã lọc được 265 thực thể/trích dẫn cho chủ đề này

Phân tích đoạn: liên_hợp_quốc (45 câu)
  Window 1/6: [API #1] Window 014R [API #2] Window 06R 
  Window 2/6: [API #3] Window 6Lỗi API (lần 1): 'NoneType' object has no attribute 'strip'
[API #4] Window 6Lỗi API (lần 2): 'NoneType' object has no attribute 'strip'
[API #5] Window 6[API #6] Window 625R 
  Window 3/6: [API #7] Window 128

C:\Users\LonelyLeaf\AppData\Local\Temp\ipykernel_21724\2200456587.py:33: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  json.dump(kg.dict(), f, ensure_ascii=False, indent=2)


Saved knowledge graph to: D:\KLTN\KLTN\knowledge_graph_historical_v2.json

PHÂN TÍCH KNOWLEDGE GRAPH
Tổng số thực thể: 490
Tổng số quan hệ: 1606

Phân bố loại thực thể:
  Địa điểm: 118
  Tổ chức: 112
  Văn kiện/Hiệp định: 45
  Sự kiện: 40
  Quốc gia: 40
  Chiến lược/Chủ trương: 40
  Hội nghị: 26
  Nhân Vật: 25
  Khái niệm: 21
  Chiến dịch/Trận đánh: 17
  Công trình: 6

Top 20 loại quan hệ phổ biến:
  thuộc_về: 98
  tham_gia: 89
  tham_gia_bởi: 86
  liên_quan_đến: 81
  diễn_ra_tại: 58
  thúc_đẩy: 36
  thông_qua: 32
  lãnh_đạo: 30
  thành_lập: 29
  thành_lập_bởi: 21
  thành_viên_của: 21
  có_quan_hệ_với: 21
  kết_thúc_với: 17
  dẫn_đến: 15
  đại_diện: 15
  đối_đầu: 14
  tại: 14
  được_thông_qua_bởi: 13
  ảnh_hưởng: 13
  hỗ_trợ: 13

Top 10 thực thể có nhiều kết nối nhất:
  Liên Xô (Quốc gia): 235 kết nối
  Liên hợp quốc (Tổ chức): 135 kết nối
  Việt Nam (Quốc gia): 108 kết nối
  Mỹ (Unknown): 84 kết nối
  Tổ chức_Liên hợp quốc (Unknown): 49 kết nối
  Pháp (Quốc gia): 44 kết nối
  ASEAN (T

C:\Users\LonelyLeaf\AppData\Local\Temp\ipykernel_21724\2200456587.py:56: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  entity.dict() for entity in kg.entities
